# **LangChain Model Prompt OutputParser**

## **Objective:**  
To demonstrate how to use LangChain’s prompt templates, chat models, and output parsers for structured response extraction. This includes making direct API calls to OpenAI, generating responses in different styles, and parsing model outputs into structured formats using Pydantic.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.

- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.

- Refer to Lesson_01
**Demo_01_Zero_Shot_Prompting.ipynb** Step 1 
for creating a virtual environment and installing the requirements.txt 

- Ensure you select the right kernel **Python (myenv)** while running the demos

---

### **Steps to perform:**
1. Set up the environment  
2. Call the direct API to OpenAI  
3. Call the API through LangChain  
4. Use the chat model  
5. Format a new message  
6. Generate a response in a new style  
7. Output parsers  
8. Use the output parser  

---


### **Step 1: Set up the environment**

- Install and import the required packages, including **OpenAI**, **LangChain**, and **Pydantic** for structured validation  
- Configure the OpenAI API key for making API calls


In [2]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import warnings
warnings.filterwarnings("ignore")


### **Step 2: Call the direct API to OpenAI**
- Create a helper function to call OpenAI’s direct API and generate responses for basic text prompts


In [3]:
client = OpenAI()   # Uses OPENAI_API_KEY automatically
def get_completion(prompt, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


- Verify the functionality of the function by asking a straightforward question, such as **What is 1+1?**.

In [4]:
print(get_completion("What is 1+1?"))


1 + 1 equals 2.


### **Step 3: Call the API through LangChain**
*   Use LangChain’s ChatOpenAI class to interact with OpenAI’s API through a high-level interface





In [5]:
chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0
)

In [6]:
template_string = """
Translate the text delimited by ``` into a style that is {style}.

Text: ```{text}```
"""

prompt_template = ChatPromptTemplate.from_template(template_string)

- Modify parameters like **customer_style** and **customer_email** to influence the tone and formality of the generated responses.

In [7]:
customer_style = "American English in a casual tone"

customer_email = """
I'm super excited about the new gaming console I bought! 
It arrived in just 2 days and I've been playing non-stop. 
Totally worth the price!
"""

customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)

### **Step 4: Use the chat model**

- Use the ChatOpenAI instance to produce a response in the specified tone and style




In [8]:
customer_response = chat.invoke(customer_messages)
print(customer_response.content)


I'm really pumped about the new gaming console I just got! It showed up in only 2 days, and I've been playing like crazy ever since. Totally worth the money!


- The model translates the text into the specified tone, demonstrating dynamic message generation.

### **Step 5: Format a new message**
- Define a new service message and format it into a conversational prompt with a different style

In [9]:
service_reply = "Hey there, we're glad you're enjoying your new gaming console. Game on!"

service_style_pirate = "a cheerful tone that speaks in English Pirate"

service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply
)

print(service_messages[0].content)



Translate the text delimited by ``` into a style that is a cheerful tone that speaks in English Pirate.

Text: ```Hey there, we're glad you're enjoying your new gaming console. Game on!```



### **Step 6: Generate a response in a new style**

- Use the chat model to generate a creative response in the specified style (English Pirate)




In [10]:
service_response = chat.invoke(service_messages)
print(service_response.content)


Ahoy matey! We be overjoyed ye be lovin' yer shiny new gaming console! Set sail fer adventure and game on, ye scallywag! Arrr!


- This step highlights how LangChain supports stylistic text transformations based on prompt parameters.

### **Step 7: Output parsers**

- Define a structured format for the model output using Pydantic



In [11]:
class ReviewInfo(BaseModel):
    gift: bool = Field(..., description="Is this item a gift?")
    delivery_days: int = Field(..., description="How many days the delivery took, or -1 if unknown.")
    price_value: list[str] = Field(..., description="Sentences describing price or value.")


In [12]:
output_parser = PydanticOutputParser(pydantic_object=ReviewInfo)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"gift": {"description": "Is this item a gift?", "title": "Gift", "type": "boolean"}, "delivery_days": {"description": "How many days the delivery took, or -1 if unknown.", "title": "Delivery Days", "type": "integer"}, "price_value": {"description": "Sentences describing price or value.", "items": {"type": "string"}, "title": "Price Value", "type": "array"}}, "required": ["gift", "delivery_days", "price_value"]}
```


### **Step 8: Use the output parser**

- Format the prompt to extract structured information from the model output





In [13]:
customer_review = """
I'm super excited about the new gaming console I bought! 
It arrived in just 2 days and I've been playing non-stop. 
Totally worth the price!
"""

review_template = """
Extract structured information from the following review.

{format_instructions}

Review:
{text}
"""

prompt = ChatPromptTemplate.from_template(review_template)


In [14]:
messages = prompt.format_messages(
    text=customer_review,
    format_instructions=format_instructions
)


In [15]:
response = chat.invoke(messages)
parsed_output = output_parser.parse(response.content)

print(parsed_output)


gift=False delivery_days=2 price_value=['Totally worth the price!']


In [16]:
print("Gift:", parsed_output.gift)
print("Delivery Days:", parsed_output.delivery_days)
print("Price Value:", parsed_output.price_value)

Gift: False
Delivery Days: 2
Price Value: ['Totally worth the price!']


- The model output is now structured and machine-readable, showcasing LangChain’s ability to handle data extraction tasks with precision.

# **Conclusion:**
By following these steps, you have successfully configured the LangChain environment, connected with OpenAI’s API, formatted and generated styled messages, and parsed structured data using Pydantic output parsers. This process demonstrates a complete workflow for managing prompts, generating responses, and extracting data programmatically with high accuracy.

---